# 05 - Resilient monitoring: new conditions, few data, broken sensors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHichem/industrial-anomaly-detection/blob/main/notebooks/05_resilient_monitoring.ipynb)

**Goal of this notebook**

Notebook 04 trained the PCA monitor on a mix of all normal batches. In a real plant, conditions change over time: a new recipe, a new control system, an ageing sensor. A monitor trained on the old conditions may raise false alarms everywhere, or miss real faults.

Here we test how **resilient** the PCA monitor is, and how to make it adapt:

1. **New conditions:** train on the fixed-recipe batches only, then apply it to operator and APC batches. Does it think everything is a fault?
2. **Adapt with few data:** add only 1, 2, 3... new APC batches to the training set. How many are needed?
3. **Broken sensor:** add a bias, a drift or a frozen value to one sensor. Does the monitor notice, and can it say which sensor?
4. **Few labels (optional):** use what the PCA learned *without labels* to help a supervised model trained on only 2 faulty batches.

We use only the PCA (fast and simple). The same tests can be run with the autoencoder of notebook 04.

**Prerequisite:** run notebook 01 once, so that the clean data is cached in Google Drive.

## 1. Loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/indpensim')
except ImportError:  # not running in Colab
    DATA_DIR = Path('data')

PROCESS_PARQUET = DATA_DIR / 'indpensim_process.parquet'
assert PROCESS_PARQUET.exists(), 'Run notebook 01 first to create the cached data.'

df = pd.read_parquet(PROCESS_PARQUET)
df['t'] = df['Time (h)'].astype('float64').round(1)  # clean time key, identical across batches
print(df.shape, '-', df['batch_id'].nunique(), 'batches')

Features naming simplification (same as notebooks 03 and 04)

In [ ]:
INPUTS = {  # Inputs of the controlled system
    'Aeration rate(Fg:L/h)': 'aeration',
    'Agitator RPM(RPM:RPM)': 'agitator_rpm',
    'Sugar feed rate(Fs:L/h)': 'sugar_feed',
    'Acid flow rate(Fa:L/h)': 'acid_flow',
    'Base flow rate(Fb:L/h)': 'base_flow',
    'Heating/cooling water flow rate(Fc:L/h)': 'cooling_water',
    'Heating water flow rate(Fh:L/h)': 'heating_water',
    'Water for injection/dilution(Fw:L/h)': 'dilution_water',
    'Air head pressure(pressure:bar)': 'head_pressure',
    'Dumped broth flow(Fremoved:L/h)': 'dumped_broth',
    'PAA flow(Fpaa:PAA flow (L/h))': 'paa_flow',
    'Oil flow(Foil:L/hr)': 'oil_flow',
    'Ammonia shots(NH3_shots:kgs)': 'nh3_shots',
}
SENSORS = {  # measurements done online
    'Dissolved oxygen concentration(DO2:mg/L)': 'DO2',
    'pH(pH:pH)': 'pH',
    'Temperature(T:K)': 'temperature',
    'carbon dioxide percent in off-gas(CO2outgas:%)': 'CO2_offgas',
    'Oxygen in percent in off-gas(O2:O2  (%))': 'O2_offgas',
    'Vessel Weight(Wt:Kg)': 'weight',
}
COMPUTED = {  # derived from measurements
    'Vessel Volume(V:L)': 'volume',
    'Generated heat(Q:kJ)': 'heat',
    'Oxygen Uptake Rate(OUR:(g min^{-1}))': 'OUR',
    'Carbon evolution rate(CER:g/h)': 'CER',
}
OTHER = {
    'Penicillin concentration(P:g/L)': 'penicillin_true',     # target
    'Fault reference(Fault_ref:Fault ref)': 'fault',          # label
}
rename = {**INPUTS, **SENSORS, **COMPUTED, **OTHER}
missing_cols = [c for c in rename if c not in df.columns]
assert not missing_cols, f'Unexpected column names: {missing_cols}'
df = df.rename(columns=rename)

MONITOR_VARS = list(INPUTS.values()) + list(SENSORS.values()) + list(COMPUTED.values())
print(len(MONITOR_VARS), 'variables available to the monitor')

## 2. Time windows (same as notebooks 03 and 04)

This time the windows are built by a **function**, because in section 5 we will modify the raw data (broken sensor) and need to rebuild the windows.

In [ ]:
WINDOW_H = 10   # window length in hours

def make_windows(data):
    """Cut each batch into 10 h windows: mean and std of each variable, and the fault label."""
    data = data.copy()
    data['window'] = (data['t'] // WINDOW_H).astype(int)   # 0 = 0-10 h, 1 = 10-20 h, ...
    groups = data.groupby(['batch_id', 'window'])

    windows = groups[MONITOR_VARS].agg(['mean', 'std'])
    windows.columns = ['_'.join(c) for c in windows.columns]   # ('DO2', 'mean') -> 'DO2_mean'
    windows = windows.fillna(0)
    windows['fault'] = groups['fault'].max()   # 1 if the fault is active in the window
    return windows.reset_index()


windows = make_windows(df)
features = windows.columns.drop(['batch_id', 'fault'])
print('Windows:', len(windows), '/', len(features), 'features')

**Batch groups.** Batches 1-90 are normal, but run with 3 different control strategies (see notebook 01). Batches 91-100 contain faults.

In [ ]:
recipe_ids   = list(range(1, 31))     # fixed recipe
operator_ids = list(range(31, 61))    # operators
apc_ids      = list(range(61, 91))    # advanced process control (Raman)
fault_ids    = list(range(91, 101))   # faulty batches

def select(ids):
    """Windows of the given batches."""
    return windows[windows['batch_id'].isin(ids)]

## 3. The PCA monitor, in 3 small functions

Same method as notebook 04: scale, compress with PCA, rebuild, and raise an alarm when the rebuild error is above the 99th percentile of the training windows.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def rebuild_error(monitor, data):
    """Rebuild error of each window (mean squared error)."""
    scaler, pca, threshold = monitor
    X = scaler.transform(data[features])
    X_rebuilt = pca.inverse_transform(pca.transform(X))
    return ((X - X_rebuilt) ** 2).mean(axis=1)


def train_monitor(train):
    """Fit the scaler and the PCA on normal windows, then set the alarm threshold."""
    scaler = StandardScaler().fit(train[features])
    pca = PCA(n_components=0.9).fit(scaler.transform(train[features]))
    monitor = (scaler, pca, None)
    threshold = np.percentile(rebuild_error(monitor, train), 99)
    return (scaler, pca, threshold)


def alarms(monitor, data):
    """1 if the window is in alarm, 0 otherwise."""
    threshold = monitor[2]
    return (rebuild_error(monitor, data) > threshold).astype(int)

## 4. New conditions: a monitor trained on the fixed recipe only

We train on 20 fixed-recipe batches (1-20) and apply the monitor to **normal batches it has never seen**: the 10 other recipe batches, then the operator and APC batches.

All these batches are normal, so every alarm is a **false alarm**. If the monitor is resilient, the alarm rate should stay around 1 % everywhere.

In [ ]:
monitor_recipe = train_monitor(select(recipe_ids[:20]))

test = select(recipe_ids[20:] + operator_ids + apc_ids).copy()
test['alarm'] = alarms(monitor_recipe, test)

group_of = {}
for b in recipe_ids:   group_of[b] = 'recipe'
for b in operator_ids: group_of[b] = 'operator'
for b in apc_ids:      group_of[b] = 'APC'
test['group'] = test['batch_id'].map(group_of)

print('False alarms (% of windows), by group:')
print((test.groupby('group')['alarm'].mean() * 100).round(1))

**Alarm rate batch by batch**, in the order the batches were run. A jump in this plot is how a drift shows up in practice.

In [ ]:
alarm_per_batch = test.groupby('batch_id')['alarm'].mean() * 100
colors = {'recipe': 'tab:blue', 'operator': 'tab:orange', 'APC': 'tab:green'}
bar_colors = [colors[group_of[b]] for b in alarm_per_batch.index]

plt.figure(figsize=(12, 3))
plt.bar(alarm_per_batch.index, alarm_per_batch.values, color=bar_colors)
plt.axhline(1, color='black', linestyle='--', label='expected (1 %)')
plt.xlabel('Batch')
plt.ylabel('Windows in alarm (%)')
plt.title('Monitor trained on recipe batches 1-20, applied to normal batches')
plt.legend()
plt.show()

**Your turn - interpretation** *(write 2-3 sentences here)*

- Does the monitor stay near 1 % on the new recipe batches? And on operator and APC batches?
- What would operators do with a monitor that behaves like this?

## 5. Adapt with few data

The plant has switched to APC. New normal APC batches arrive one by one. How many do we need to add to the training set before the monitor works again?

- **Adaptation pool:** APC batches 61-70, added one after the other.
- **Fixed test set:** APC batches 71-90 (false alarms) and the 10 faulty batches (detection).

We compare two strategies:
- **old + new:** keep the 20 recipe batches and add the new APC batches;
- **new only:** forget the old data and train on the new APC batches only.

In [ ]:
adapt_pool = apc_ids[:10]                  # 61-70
apc_test = select(apc_ids[10:])            # 71-90, normal
faults = select(fault_ids)
fault_windows = faults[faults['fault'] == 1]

old_ids = recipe_ids[:20]

results = []
for n in [0, 1, 2, 3, 5, 10]:
    new_ids = adapt_pool[:n]

    monitor = train_monitor(select(old_ids + new_ids))
    results.append({'new batches': n, 'strategy': 'old + new',
                    'False alarms (%)': alarms(monitor, apc_test).mean() * 100,
                    'Detection (%)': alarms(monitor, fault_windows).mean() * 100})

    if n > 0:
        monitor = train_monitor(select(new_ids))
        results.append({'new batches': n, 'strategy': 'new only',
                        'False alarms (%)': alarms(monitor, apc_test).mean() * 100,
                        'Detection (%)': alarms(monitor, fault_windows).mean() * 100})

results = pd.DataFrame(results).round(1)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for strategy in ['old + new', 'new only']:
    r = results[results['strategy'] == strategy]
    axes[0].plot(r['new batches'], r['False alarms (%)'], marker='o', label=strategy)
    axes[1].plot(r['new batches'], r['Detection (%)'], marker='o', label=strategy)

axes[0].set_title('False alarms on new APC batches')
axes[1].set_title('Detection of faults')
for ax in axes:
    ax.set_xlabel('Number of new APC batches in training')
    ax.set_ylabel('%')
    ax.legend()
plt.show()

**Your turn - interpretation** *(write 2-3 sentences here)*

- How many new batches are needed to bring false alarms back near 1 %?
- Which strategy is better when very few new batches are available? Does detection suffer?

## 6. Broken sensor

Sensors age, drift or get stuck. We take the normal APC test batches (71-90) and break **one sensor** after 100 h, in 3 ways:

- **bias:** a constant offset;
- **drift:** an offset that grows slowly with time;
- **frozen:** the sensor keeps showing the same value.

The size of the problem is given in "typical variations" of the sensor (its standard deviation). We use the monitor adapted in section 5 (old + 10 new APC batches).

Here an alarm is **good news**: the monitor noticed the broken sensor.

In [ ]:
monitor = train_monitor(select(old_ids + adapt_pool))

SENSOR = 'DO2'     # try also 'pH', 'temperature', 'CO2_offgas'
START_H = 100      # the sensor breaks at 100 h
SIZE = 1           # size of the problem, in standard deviations of the sensor

clean = df[df['batch_id'].isin(apc_ids[10:])].copy()
step = SIZE * clean[SENSOR].std()
after = clean['t'] >= START_H

bias = clean.copy()
bias.loc[after, SENSOR] += step

drift = clean.copy()
drift.loc[after, SENSOR] += step * (drift.loc[after, 't'] - START_H) / 100   # + SIZE std every 100 h

frozen = clean.copy()
frozen.loc[after, SENSOR] = np.nan
frozen[SENSOR] = frozen.groupby('batch_id')[SENSOR].ffill()   # stuck at the last value before 100 h

rows = []
for name, data in {'no problem': clean, 'bias': bias, 'drift': drift, 'frozen': frozen}.items():
    w = make_windows(data)
    w = w[w['window'] >= START_H // WINDOW_H]   # only windows after the sensor broke
    rows.append({'Sensor': name, 'Windows in alarm (%)': alarms(monitor, w).mean() * 100})

pd.DataFrame(rows).round(1)

**Which sensor is responsible?** For the windows in alarm, we look at which features are rebuilt worst. This is what an operator needs: not only "something is wrong" but "look at this sensor".

In [ ]:
scaler, pca, threshold = monitor

w = make_windows(bias)
w = w[w['window'] >= START_H // WINDOW_H]
w = w[alarms(monitor, w) == 1]

X = scaler.transform(w[features])
X_rebuilt = pca.inverse_transform(pca.transform(X))
contribution = pd.Series(((X - X_rebuilt) ** 2).mean(axis=0), index=features)

contribution.sort_values().tail(10).plot.barh(figsize=(7, 4), title=f'Features rebuilt worst ({SENSOR} bias)')
plt.show()

**Your turn - interpretation** *(write 2-3 sentences here)*

- Which sensor problem is the easiest to catch? The hardest? (Change `SIZE` to 0.5 and 2 to see the limits.)
- Does the contribution plot point to the right sensor?

## 7. Few labels (optional): reuse what the PCA learned without labels

In a real plant, labelled faults are rare. Here we pretend that only **2 faulty batches** (91 and 92) are labelled.

- **(a) Random Forest on the raw features:** trained on the 47 features, with only 2 faulty batches.
- **(b) Random Forest on the PCA features:** the PCA was trained on normal batches **without any label**. We give the Random Forest the compressed window (PCA components) and the rebuild error.

Idea (b) is the spirit of **self-supervised learning**: first learn the structure of the data without labels, then use a few labels on top.

Test: APC batches 71-90 (false alarms) and the 8 faulty batches 93-100 (detection).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

labelled_faults = fault_ids[:2]   # 91, 92
unseen_faults   = fault_ids[2:]   # 93-100

train_lab = select(old_ids + adapt_pool + labelled_faults)
test_lab  = select(apc_ids[10:] + unseen_faults)

def pca_features(data):
    """What the PCA learned without labels: compressed window + rebuild error."""
    X = scaler.transform(data[features])
    scores = pca.transform(X)
    error = ((X - pca.inverse_transform(scores)) ** 2).mean(axis=1)
    return np.column_stack([scores, error])

def show(name, pred):
    false_alarms = pred[test_lab['fault'] == 0].mean() * 100
    detection    = pred[test_lab['fault'] == 1].mean() * 100
    print(f'{name}: false alarms {false_alarms:.1f} % / detection {detection:.1f} %')

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0)

rf.fit(train_lab[features], train_lab['fault'])
show('(a) raw features', rf.predict(test_lab[features]))

rf.fit(pca_features(train_lab), train_lab['fault'])
show('(b) PCA features', rf.predict(pca_features(test_lab)))

show('(c) PCA alarm alone, no labels', alarms(monitor, test_lab))

**Your turn - interpretation** *(write 2-3 sentences here)*

- With only 2 labelled faulty batches, which option is best?
- Is it worth using labels at all, compared with the PCA alarm alone (c)?

## 8. Conclusions: a strategy for resilient monitoring

*(Hichem - 4 to 5 bullet points. Suggested structure:)*

1. **Learn "normal" without labels** (PCA / autoencoder): it also catches faults never seen before.
2. **Watch the alarm rate over time:** a jump on normal batches means the conditions changed (drift), not that everything is faulty.
3. **Adapt with few data:** retrain with the first new normal batches; say how many were needed and which strategy worked.
4. **Point to the sensor:** the contribution plot tells the operator where to look, and catches broken sensors.
5. **Use the few labels on top** of what was learned without labels, if they help.